In [9]:
import pandas as pd
import json
import re
from sentence_transformers import SentenceTransformer

d:\FYP\29-3-25\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1) Load the raw JD file
df_jd = pd.read_csv('../../data/processed/job_descriptions.csv', dtype=str)

# 2) Drop rows with missing jd_id or jd_text
df_jd = df_jd.dropna(subset=['jd_id', 'jd_text'])

In [5]:
df_jd.head()

,jd_id,role,jd_text,extracted_skills,Unnamed: 4
0,ds_jd_01,Data Scientist,\r\n**Data Scientist - Southeast Asia (Philipp...,"Machine Learning, Predictive Modeling, Data Pi...",NaN
1,ds_jd_02,Data Scientist,\r\n### **Data Scientist**\r\n\r\n**Job Purpos...,"Machine Learning, Python, Data Visualization, ...",NaN
2,ds_jd_03,Data Scientist,\r\n### **Python Data Scientist / Analyst (Rem...,"Python, Data Analysis, Artificial Intelligence...",NaN
3,ds_jd_04,Data Scientist,\r\n### **Data Scientist**\r\n\r\n**What You'l...,"Machine Learning, Python, Deep Learning, Data ...",NaN
4,ds_jd_05,Data Scientist,\r\n\r\n### **Data Scientist**\r\n\r\nJoin **P...,"Machine Learning, Python, Data Analysis, Stati...",NaN


In [6]:
df_jd['skills_list'] = df_jd['extracted_skills'].str.split(', ').apply(lambda lst: [s.lower() for s in lst])

# Compute unique skills
unique_skills = sorted({skill for lst in df_jd['skills_list'] for skill in lst})

# Build summary
summary_df = pd.DataFrame({
    "Total JDs": [len(df_jd)],
    "Unique Skills Count": [len(unique_skills)],
    "Unique Skills": [", ".join(unique_skills)]
})

summary_df.head()

,Total JDs,Unique Skills Count,Unique Skills
0,40,109,".net, agile, agile devops, ai strategy, angula..."


In [7]:
df_jd = df_jd.loc[:, ~df_jd.columns.str.contains('^Unnamed')]
required_cols = ['jd_id', 'role', 'jd_text', 'extracted_skills']
df_jd = df_jd.dropna(subset=required_cols)


In [10]:
def clean_jd_text(text):
    text = text or ''
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove markdown symbols
    text = re.sub(r'[#*_>`\-]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'[\r\n]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [11]:
df_jd['jd_text_clean'] = df_jd['jd_text'].apply(clean_jd_text)

In [12]:
def parse_skills(sk):
    if pd.isna(sk):
        return []
    return [skill.strip().lower() for skill in sk.split(',') if skill.strip()]

df_jd['skills_list'] = df_jd['extracted_skills'].apply(parse_skills)


In [13]:
df_jd['skills_text'] = df_jd['skills_list'].apply(lambda lst: ' '.join(lst))

# 6. Embed skills_text using SBERT
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(df_jd['skills_text'].tolist())
df_jd['skills_vector'] = embeddings.tolist()


In [14]:
display_df = df_jd[['jd_id', 'role', 'jd_text_clean', 'skills_list', 'skills_vector']]

In [15]:
display_df.head()

,jd_id,role,jd_text_clean,skills_list,skills_vector
0,ds_jd_01,Data Scientist,Data Scientist Southeast Asia (Philippines) We...,"[machine learning, predictive modeling, data p...","[-0.11037430912256241, -0.076742023229599, 0.0..."
1,ds_jd_02,Data Scientist,Data Scientist Job Purpose: We're seeking a sk...,"[machine learning, python, data visualization,...","[0.002306657610461116, -0.01614946685731411, 0..."
2,ds_jd_03,Data Scientist,Python Data Scientist / Analyst (Remote) Exper...,"[python, data analysis, artificial intelligenc...","[-0.03032637946307659, 0.053438253700733185, 0..."
3,ds_jd_04,Data Scientist,Data Scientist What You'll Do: Build machine l...,"[machine learning, python, deep learning, data...","[-0.024929551407694817, -0.04848936200141907, ..."
4,ds_jd_05,Data Scientist,"Data Scientist Join Paradigm , a rapidly growi...","[machine learning, python, data analysis, stat...","[-0.06652437150478363, -0.014627852477133274, ..."


In [18]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [19]:
embeddings = embedder.encode(display_df['jd_text_clean'].tolist(), show_progress_bar=True)

# 4) Attach to your DataFrame
display_df['jd_text_embedding'] = embeddings.tolist()


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.52it/s]
C:\Users\ASUS\AppData\Local\Temp\ipykernel_33712\3335394640.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  display_df['jd_text_embedding'] = embeddings.tolist()


In [20]:
display_df.head()

,jd_id,role,jd_text_clean,skills_list,skills_vector,jd_text_embedding
0,ds_jd_01,Data Scientist,Data Scientist Southeast Asia (Philippines) We...,"[machine learning, predictive modeling, data p...","[-0.11037430912256241, -0.076742023229599, 0.0...","[-0.0031088069081306458, -0.06425159424543381,..."
1,ds_jd_02,Data Scientist,Data Scientist Job Purpose: We're seeking a sk...,"[machine learning, python, data visualization,...","[0.002306657610461116, -0.01614946685731411, 0...","[-0.017808740958571434, -0.019467582926154137,..."
2,ds_jd_03,Data Scientist,Python Data Scientist / Analyst (Remote) Exper...,"[python, data analysis, artificial intelligenc...","[-0.03032637946307659, 0.053438253700733185, 0...","[-0.04353145882487297, -0.008451883681118488, ..."
3,ds_jd_04,Data Scientist,Data Scientist What You'll Do: Build machine l...,"[machine learning, python, deep learning, data...","[-0.024929551407694817, -0.04848936200141907, ...","[-0.010062647052109241, -0.027305958792567253,..."
4,ds_jd_05,Data Scientist,"Data Scientist Join Paradigm , a rapidly growi...","[machine learning, python, data analysis, stat...","[-0.06652437150478363, -0.014627852477133274, ...","[-0.05086949095129967, -0.0799584835767746, 0...."


In [21]:
display_df.to_csv('../../data/test-final/jds_clean.csv', index=False)
print("Dataset  saved as 'jds_clean.csv'")

Dataset  saved as 'jds_clean.csv'
